# 02 - Baseline Model: TF-IDF + Logistic Regression

**Objective:** Establish a strong baseline with interpretable features before moving to deep learning.

**Target:** >85% accuracy on 4-class document classification

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries loaded.")

In [ ]:
DATA_DIR = '../data'

df = pd.read_csv(f'{DATA_DIR}/documents.csv')

# Text preprocessing
import re
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return ' '.join(text.split())

df['text_clean'] = df['text'].apply(preprocess)

# Encode labels
label_map = {cat: idx for idx, cat in enumerate(df['category'].unique())}
df['label'] = df['category'].map(label_map)

print(f"Classes: {label_map}")
print(f"Total samples: {len(df)}")

## TF-IDF Vectorization

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    df['text_clean'], df['label'],
    test_size=0.2, random_state=42, stratify=df['label']
)

# TF-IDF
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.95,
    stop_words='english'
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f"TF-IDF features: {X_train_tfidf.shape[1]}")
print(f"Train: {X_train_tfidf.shape[0]}, Test: {X_test_tfidf.shape[0]}")

## Logistic Regression Model

In [ ]:
# Train model
lr = LogisticRegression(max_iter=1000, random_state=42, C=1.0)
lr.fit(X_train_tfidf, y_train)

# Predict
y_pred = lr.predict(X_test_tfidf)

# Metrics
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.3f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=list(label_map.keys())))

# Cross-validation
cv_scores = cross_val_score(lr, X_train_tfidf, y_train, cv=5)
print(f"\nCV Accuracy: {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})")

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
          xticklabels=list(label_map.keys()),
          yticklabels=list(label_map.keys()))
plt.title('Confusion Matrix - TF-IDF + Logistic Regression')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('../figures/nlp_02_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ../figures/nlp_02_confusion_matrix.png")

## Feature Importance

In [ ]:
# Top features per class
feature_names = tfidf.get_feature_names_out()
coef = lr.coef_

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, (cat, label_id) in enumerate(label_map.items()):
    top_idx = np.argsort(coef[label_id])[-10:][::-1]
    ax = axes[idx // 2, idx % 2]
    ax.barh(range(10), coef[label_id][top_idx], color='steelblue')
    ax.set_yticks(range(10))
    ax.set_yticklabels([feature_names[i] for i in top_idx], fontsize=8)
    ax.set_title(f'Top Features: {cat}')
    ax.invert_yaxis()

plt.tight_layout()
plt.savefig('../figures/nlp_02_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ../figures/nlp_02_feature_importance.png")

## Model Summary

- **Accuracy:** {:.1%}
- **CV Accuracy:** {:.1%} (+/- {:.1%})
- **Best Features:** Distinctive n-grams per category
- **Interpretability:** High - coefficients directly explain predictions

Next: BERT Fine-Tuning (Notebook 03)